# HyperSense — Deployment Pipeline

This notebook prepares the final XGBoost model for deployment within the HyperSense screening application.

Objectives:

1. Export final trained model
2. Save deployment threshold
3. Validate prediction pipeline
4. Validate SHAP explanation pipeline
5. Prepare assets for Streamlit deployment

The final output of this notebook will be a deployment-ready machine learning artifact and supporting assets.

In [20]:
import pandas as pd
import numpy

import joblib
from pathlib import Path

import shap


In [13]:
# Create folders

Path("../models").mkdir(exist_ok=True)
Path("../app").mkdir(exist_ok=True)

print("Deployment folders created.")

Deployment folders created.


In [14]:
# Save threshold

SCREENING_THRESHOLD = 0.353

joblib.dump(
    SCREENING_THRESHOLD,
    "../models/hypersense_threshold.pkl"
)

print("Threshold saved successfully.")

Threshold saved successfully.


In [15]:
# Reload and test

loaded_model = joblib.load(
    "../models/hypersense_xgb.pkl"
)

loaded_threshold = joblib.load(
    "../models/hypersense_threshold.pkl"
)

print(loaded_threshold)

0.353


In [19]:
# Test single prediction

sample_patient = pd.DataFrame({
    "age": [58],
    "gender": [1],
    "residence": [1],
    "educational_level": [3],
    "tobacco_use": [0]
})

# Predict:
risk_probability = loaded_model.predict_proba(
    sample_patient
)[0, 1]

print(
    f"Predicted risk: {risk_probability:.3f}"
)

if risk_probability >= loaded_threshold:
    print("High Risk")
else:
    print("Low Risk")

Predicted risk: 0.795
High Risk


In [21]:
# Test SHAP in deployment environ

# Create explainer
explainer = shap.TreeExplainer(
    loaded_model
)

# Generate SHAP values
shap_values = explainer.shap_values(
    sample_patient
)

print(shap_values)

[[ 1.248624   -0.01015806  0.04962574  0.43611857 -0.00383138]]
